In [1]:
import numpy as np

mask_path = "/home/nikitachernysh/storage/Projects/lidar-archaeology-segmentation/data/processed/mounds_mask_shadowed.npy"
#probs_path = "../evaluation_results/DEM_full_prediction_map.npy"
#probs_path = "/home/nikitachernysh/storage/Projects/lidar-archaeology-segmentation/experiments/0.80(5)_norm_shadow_resnet101(ImageNet1k_V2)_128(32)_BCE+DICE0.5_rednone_b16w16e50_l1e-4/DEM_full_prediction_map.npy"
probs_path = "/home/nikitachernysh/storage/Projects/lidar-archaeology-segmentation/experiments/0.80(5)_norm_shadow_resnet101_128(32)_BCE+DICE0.5_rednone_b16w16e50_l1e-4/DEM_full_prediction_map.npy"

mask = np.load(mask_path)
probs = np.load(probs_path)

In [3]:
from skimage.morphology import opening, closing, disk
import numpy as np

def calculate_metrics(probs, targets, valid=None, threshold=0.8, morph=False):
    """Calculate metrics for binary semantic segmentation.
    
    Args:
        probs: Model output logits (after sigmoid) of shape (B, 1, H, W)
        targets: Ground truth masks of shape (B, 1, H, W)
        valid: Optional mask indicating valid pixels
        threshold: Threshold for binary prediction
        
    Returns:
        Dictionary containing various segmentation metrics
    """

    def compute_metrics(pred_bin, gt):
        tp = np.logical_and(pred_bin == 1, gt == 1).sum()
        fp = np.logical_and(pred_bin == 1, gt == 0).sum()
        fn = np.logical_and(pred_bin == 0, gt == 1).sum()
        tn = np.logical_and(pred_bin == 0, gt == 0).sum()

        iou = tp / (tp + fp + fn + 1e-8)
        acc = (tp + tn) / (tp + tn + fp + fn + 1e-8)
        mor10r = tp / (tp + fn + 10*fp + 1e-8)
        return iou, acc, mor10r

    best_t, best_iou, best_acc, best_mor = 0, 0, 0, 0
    thresholds = np.linspace(0.4, 0.7, 30)

    for t in thresholds:
        pred_bin = (probs > t).astype(np.uint8)
        iou, acc, mor = compute_metrics(pred_bin, targets)
        if iou > best_iou:
            best_t, best_iou, best_acc, best_mor = t, iou, acc, mor

    print(f"Optimal threshold: {best_t:.2f}")
    print(f"IoU={best_iou:.3f}, Accuracy={best_acc:.3f}, MOR10R={best_mor:.3f}")

    # Create binary predictions
    preds = (probs > best_t).astype(np.uint8)

    if morph:
        best_d, best_iou, best_acc, best_mor = 0, 0, 0, 0
        for d in [1,2,3,4,5]:
            test_opening = compute_metrics(opening(preds, disk(d)), targets)
            test_closing = compute_metrics(closing(preds, disk(d)), targets)
            test_opening_closing = compute_metrics(closing(opening(preds, disk(d)), disk(d)), targets)
            test_closing_opening = compute_metrics(opening(closing(preds, disk(d)), disk(d)), targets)
    
            if test_opening[0] > best_iou:
                best_d, best_iou, best_acc, best_mor = d, test_opening[0], test_opening[1], test_opening[2]
            if test_closing[0] > best_iou:
                best_d, best_iou, best_acc, best_mor = d, test_closing[0], test_closing[1], test_closing[2]
            if test_opening_closing[0] > best_iou:
                best_d, best_iou, best_acc, best_mor = d, test_opening_closing[0], test_opening_closing[1], test_opening_closing[2]
            if test_closing_opening[0] > best_iou:
                best_d, best_iou, best_acc, best_mor = d, test_closing_opening[0], test_closing_opening[1], test_closing_opening[2]
    
        print(f"Optimal disk size: {best_d}")
        print(f"IoU={best_iou:.3f}, Accuracy={best_acc:.3f}, MOR10R={best_mor:.3f}")
        
    tp = np.logical_and(preds == 1, targets == 1).sum()
    fp = np.logical_and(preds == 1, targets == 0).sum()
    fn = np.logical_and(preds == 0, targets == 1).sum()
    tn = np.logical_and(preds == 0, targets == 0).sum()
    
    # Pixel-level metrics
    eps = 1e-10
    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    
    # Class-wise metrics
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * (precision * recall) / (precision + recall + eps)
    
    # IoU (Jaccard Index) for each class
    iou_positive = tp / (tp + fp + fn + eps)
    
    return {
        # Overall metrics
        'accuracy': accuracy,   
        
        # Class-specific metrics
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'iou': iou_positive,
        
        # Additional metrics
        'specificity': tn / (tn + fp + eps),  # True Negative Rate
    }

In [4]:
from scipy.ndimage import uniform_filter

smoothed_5 = uniform_filter(probs, size=5)
#smoothed_10 = uniform_filter(probs, size=10)
#smoothed_15 = uniform_filter(probs, size=15)

#print("Without smoothing:")
#metrics = calculate_metrics(probs, mask)
#for metric in metrics:
#    print(f"{metric}: {metrics[metric]:.6f}")

print("-" * 40)
print("With smoothing:")
print("5x5")
metrics_5 = calculate_metrics(smoothed_5, mask)
for metric in metrics_5:
    print(f"{metric}: {metrics_5[metric]:.6f}")
#print("\n10x10")
#metrics_10 = calculate_metrics(smoothed_10, mask)
#for metric in metrics_10:
#    print(f"{metric}: {metrics_10[metric]:.6f}")
#print("\n15x15")
#metrics_15 = calculate_metrics(smoothed_15, mask)
#for metric in metrics_15:
#    print(f"{metric}: {metrics_15[metric]:.6f}")


----------------------------------------
With smoothing:
5x5
Optimal threshold: 0.54
IoU=0.379, Accuracy=0.998, MOR10R=0.105
accuracy: 0.997623
precision: 0.565320
recall: 0.534123
f1: 0.549279
iou: 0.378624
specificity: 0.998883


In [18]:
from scipy.ndimage import uniform_filter

smoothed_5 = uniform_filter(probs, size=5)
smoothed_10 = uniform_filter(probs, size=10)
smoothed_15 = uniform_filter(probs, size=15)

print("Without smoothing:")
metrics = calculate_metrics(probs, mask, morph=True)
for metric in metrics:
    print(f"{metric}: {metrics[metric]:.6f}")

print("-" * 40)
print("With smoothing:")
print("5x5")
metrics_5 = calculate_metrics(smoothed_5, mask, morph=True)
for metric in metrics_5:
    print(f"{metric}: {metrics_5[metric]:.6f}")
print("\n10x10")
metrics_10 = calculate_metrics(smoothed_10, mask, morph=True)
for metric in metrics_10:
    print(f"{metric}: {metrics_10[metric]:.6f}")
print("\n15x15")
metrics_15 = calculate_metrics(smoothed_15, mask, morph=True)
for metric in metrics_15:
    print(f"{metric}: {metrics_15[metric]:.6f}")

Without smoothing:
Optimal threshold: 0.61
IoU=0.485, Accuracy=0.997, MOR10R=0.150
Optimal disk size: 1
IoU=0.486, Accuracy=0.997, MOR10R=0.153
accuracy: 0.997161
precision: 0.660420
recall: 0.646892
f1: 0.653586
iou: 0.485427
specificity: 0.998617
----------------------------------------
With smoothing:
5x5
Optimal threshold: 0.53
IoU=0.489, Accuracy=0.997, MOR10R=0.147
Optimal disk size: 2
IoU=0.489, Accuracy=0.997, MOR10R=0.147
accuracy: 0.997150
precision: 0.655232
recall: 0.657541
f1: 0.656384
iou: 0.488521
specificity: 0.998562

10x10
Optimal threshold: 0.45
IoU=0.465, Accuracy=0.997, MOR10R=0.136
Optimal disk size: 1
IoU=0.465, Accuracy=0.997, MOR10R=0.136
accuracy: 0.996968
precision: 0.633370
recall: 0.635614
f1: 0.634490
iou: 0.464654
specificity: 0.998470

15x15
Optimal threshold: 0.37
IoU=0.427, Accuracy=0.997, MOR10R=0.116
Optimal disk size: 1
IoU=0.427, Accuracy=0.997, MOR10R=0.116
accuracy: 0.996616
precision: 0.588258
recall: 0.608469
f1: 0.598193
iou: 0.426730
specific